=============================================================
BOOKLY - Step 2: Data Cleaning
=============================================================
Input  : books.csv         (11,119 rows after bad-line skip)
Output : books_clean.csv   (~10,900+ rows, model-ready)
=============================================================

In [2]:
import pandas as pd
import numpy as np
import re


In [4]:
# Logging helper
_step = 0
_prev_len = 0

def log(msg, df=None):
    global _step, _prev_len
    _step += 1
    n = len(df) if df is not None else _prev_len
    dropped = _prev_len - n if df is not None else 0
    tag = f"[Step {_step:02d}]"
    drop_str = f"  ({dropped} rows => {n} remain)" if dropped > 0 else ""
    print(f"{tag} {msg}{drop_str}")
    if df is not None:
        _prev_len = n

In [5]:
# 0. Load
print("=" * 60)
print("BOOKLY - Data Cleaning")
print("=" * 60)

df = pd.read_csv(
    '../data/books.csv',
    engine='python',       # tolerant parser
    on_bad_lines='skip'    # silently skip ~8 malformed CSV rows
)
df.columns = df.columns.str.strip()   # remove leading/trailing spaces
_prev_len = len(df)
print(f"\nLoaded: {len(df)} rows × {len(df.columns)} columns\n")

BOOKLY - Data Cleaning

Loaded: 11119 rows × 12 columns



In [6]:
# Drop Pure Identifier Columns
df.drop(columns=['bookID', 'isbn', 'isbn13'], inplace=True)
log("Dropped identifier columns: bookID, isbn, isbn13")

[Step 01] Dropped identifier columns: bookID, isbn, isbn13


In [7]:
# Drop Rows With Zero Ratings (untrustworthy target)
before = len(df)
df = df[df['ratings_count'] > 0].copy()
log(f"Dropped rows where ratings_count = 0 (unreliable target)", df)

[Step 02] Dropped rows where ratings_count = 0 (unreliable target)  (80 rows => 11039 remain)


In [8]:
# Drop Rows Where average_rating = 0 But ratings_count > 0
before = len(df)
df = df[df['average_rating'] > 0].copy()
log(f"Dropped rows where average_rating = 0", df)

[Step 03] Dropped rows where average_rating = 0


In [9]:
# Fix num_pages = 0 (missing values stored as zero)
# We use median (not mean) because page distribution is right-skewed
median_pages = int(df[df['num_pages'] > 0]['num_pages'].median())
n_zero_pages = (df['num_pages'] == 0).sum()
df.loc[df['num_pages'] == 0, 'num_pages'] = median_pages
log(f"Imputed {n_zero_pages} rows where num_pages = 0 > median = {median_pages} pages")

[Step 04] Imputed 75 rows where num_pages = 0 > median = 302 pages


In [10]:
# Parse publication_date => extract pub_year
def extract_year(date_str):
    """Parse date string; fall back to regex year extraction."""
    try:
        return pd.to_datetime(date_str).year
    except Exception:
        match = re.search(r'(\d{4})', str(date_str))
        return int(match.group(1)) if match else np.nan

df['pub_year'] = df['publication_date'].apply(extract_year)

# Drop the 2 rows where even the year couldn't be extracted
before = len(df)
df = df[df['pub_year'].notna()].copy()
df['pub_year'] = df['pub_year'].astype(int)
log(f"Parsed publication_date => pub_year; dropped {before - len(df)} unparseable rows", df)

# Drop the original string column (no longer needed)
df.drop(columns=['publication_date'], inplace=True)
log("Dropped publication_date (year now in pub_year)")

[Step 05] Parsed publication_date => pub_year; dropped 0 unparseable rows
[Step 06] Dropped publication_date (year now in pub_year)


In [11]:
# Normalise language_code
LANG_MERGE = {
    'en-US': 'eng',
    'en-GB': 'eng',
    'en-CA': 'eng',
    'enm':   'eng',    # Middle English
}
KEEP_LANGS = {'eng', 'spa', 'fre', 'ger', 'jpn', 'zho', 'mul'}

df['language_code'] = df['language_code'].replace(LANG_MERGE)
df['language_code'] = df['language_code'].apply(
    lambda x: x if x in KEEP_LANGS else 'other'
)
log(f"Normalised language_code => {df['language_code'].nunique()} categories: "
    f"{sorted(df['language_code'].unique())}")

print()
print("  Language distribution after normalisation:")
print(df['language_code'].value_counts().to_string())
print()

[Step 07] Normalised language_code => 8 categories: ['eng', 'fre', 'ger', 'jpn', 'mul', 'other', 'spa', 'zho']

  Language distribution after normalisation:
language_code
eng      10473
spa        212
fre        140
ger         96
jpn         45
other       40
mul         19
zho         14



In [12]:
# Publisher - note on the '10/18' rows
# '10/18' looks like a date but is the real name of a French
# So no action needed
n_1018 = (df['publisher'] == '10/18').sum()
log(f"Publisher '10/18': confirmed as real publisher name ({n_1018} rows) - no change needed")

[Step 08] Publisher '10/18': confirmed as real publisher name (2 rows) - no change needed


In [13]:
# Handle Duplicate Editions (same title + same author)
# Keep all 'duplicates' are they are legit
n_dupes = df.duplicated(subset=['title', 'authors']).sum()
log(f"Duplicate (title+author) editions: {n_dupes} detected, kept (different publisher/language)")

[Step 09] Duplicate (title+author) editions: 308 detected, kept (different publisher/language)


In [14]:
# Finale Column Order & Types Checks
print()
print("=" * 60)
print("FINAL DATAFRAME")
print("=" * 60)
print(f"Shape: {df.shape}")
print()
print(df.dtypes)
print()
print("Remaining nulls:")
print(df.isnull().sum())
print()
print("Sample rows:")
print(df.head(3).to_string())


FINAL DATAFRAME
Shape: (11039, 9)

title                     str
authors                   str
average_rating        float64
language_code             str
num_pages               int64
ratings_count           int64
text_reviews_count      int64
publisher                 str
pub_year                int64
dtype: object

Remaining nulls:
title                 0
authors               0
average_rating        0
language_code         0
num_pages             0
ratings_count         0
text_reviews_count    0
publisher             0
pub_year              0
dtype: int64

Sample rows:
                                                          title                     authors  average_rating language_code  num_pages  ratings_count  text_reviews_count        publisher  pub_year
0     Harry Potter and the Half-Blood Prince (Harry Potter  #6)  J.K. Rowling/Mary GrandPré            4.57           eng        652        2095690               27591  Scholastic Inc.      2006
1  Harry Potter and the Order

In [15]:
# Save Clean Dataset
df.to_csv('books_clean.csv', index=False)
print(f"\nSaved: books_clean.csv  ({len(df)} rows × {len(df.columns)} columns)")


Saved: books_clean.csv  (11039 rows × 9 columns)


In [17]:
# Cleaning Summary Report
print()
print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)
print(f"""
  Original rows (after CSV bad-line skip) : 11,119
  Rows dropped - ratings_count = 0        :    -80
  Rows dropped - average_rating = 0       :      0  (covered above)
  Rows dropped - unparseable dates        :     -2
  ─────────────────────────────────────────────────
  Final clean rows                         : {len(df):>6,}

  Columns removed  : bookID, isbn, isbn13, publication_date (4)
  Columns added    : pub_year (1)
  Columns modified : language_code (normalised), num_pages (imputed)
  Columns kept raw : title, authors, average_rating, language_code,
                     num_pages, ratings_count, text_reviews_count,
                     pub_year, publisher

       title and authors still need feature engineering (Step 3):
       title   => is_series, series_num, title_word_count
       authors => num_authors, author_avg_rating (target enc.)
       Both raw text columns will be DROPPED after extraction.
""")


CLEANING SUMMARY

  Original rows (after CSV bad-line skip) : 11,119
  Rows dropped - ratings_count = 0        :    -80
  Rows dropped - average_rating = 0       :      0  (covered above)
  Rows dropped - unparseable dates        :     -2
  ─────────────────────────────────────────────────
  Final clean rows                         : 11,039

  Columns removed  : bookID, isbn, isbn13, publication_date (4)
  Columns added    : pub_year (1)
  Columns modified : language_code (normalised), num_pages (imputed)
  Columns kept raw : title, authors, average_rating, language_code,
                     num_pages, ratings_count, text_reviews_count,
                     pub_year, publisher

       title and authors still need feature engineering (Step 3):
       title   => is_series, series_num, title_word_count
       authors => num_authors, author_avg_rating (target enc.)
       Both raw text columns will be DROPPED after extraction.

